In [1]:
import os
import pandas as pd

## RIS cameras distributions

In [ ]:
shot_usage = pd.read_csv('./data/shot_usageNEW.csv', index_col=0)

distribution_df = pd.DataFrame(index=shot_usage.index, columns=['n_imgs_RIS1', 'n_imgs_RIS2'])

for shot_number in shot_usage.index:
    for _, _, imgs in os.walk(f'./imgs/{shot_number}'):
        n_imgs_RIS1 = 0
        n_imgs_RIS2 = 0
        for img in imgs:
            if 'RIS1' in img and shot_usage.at[shot_number, 'used_for_ris1']:
                n_imgs_RIS1 += 1
            elif 'RIS2' in img and shot_usage.at[shot_number, 'used_for_ris2']:
                n_imgs_RIS2 += 1
        distribution_df.at[shot_number, 'n_imgs_RIS1'] = n_imgs_RIS1
        distribution_df.at[shot_number, 'n_imgs_RIS2'] = n_imgs_RIS2

non_zero_dist_df = distribution_df[~(distribution_df.any(axis=1) == 0)]
non_zero_dist_df.sort_values(by='n_imgs_RIS2', ascending=False).sum()

In [15]:
# Set the directory path
directory = '/compass/Shared/Users/bogdanov/ml_tokamak/imgs'

# Initialize a counter for the number of files
file_count = 0

# Walk through all subfolders and count the files
for root, dirs, files in os.walk(directory):
    file_count += len(files)

# Print the total number of files
print(f'Total number of files: {file_count}')

Total number of files: 378675


### Mode distribution

In [4]:
from pathlib import Path
from utils import confinement_mode_classifier as cmc
import numpy as np

ris_option = 'RIS1'

shot_usage = pd.read_csv(f'/compass/Shared/Users/bogdanov/ml_tokamak/data/shot_usageNEW.csv')
shot_for_ris = shot_usage[shot_usage['used_for_ris2'] if ris_option == 'RIS2' else shot_usage['used_for_ris1']]
shot_numbers = shot_for_ris['shot']
shots_for_testing = shot_for_ris[shot_for_ris['used_as'] == 'test']['shot']
shots_for_validation = shot_for_ris[shot_for_ris['used_as'] == 'val']['shot']
shots_for_training = shot_for_ris[shot_for_ris['used_as'] == 'train']['shot']

path = Path(os.getcwd())

ModuleNotFoundError: No module named 'utils'

In [ ]:
shot_df, test_df, val_df, train_df = cmc.load_and_split_dataframes(path,shot_numbers, shots_for_training, shots_for_testing, 
                                                                    shots_for_validation, use_ELMS=True, ris_option=ris_option,
                                                                    exponential_elm_decay=False)

dist_df = pd.DataFrame({'train_df': train_df['mode'].value_counts().values, 
                        'val_df': val_df['mode'].value_counts().values, 
                        'test_df': test_df['mode'].value_counts().values}, 
                       index=['L-mode', 'H-mode', 'ELM'])
display(dist_df)

display(dist_df.sum().sum())

### How many continious ELM has ResNet recognized?

In [1]:
import os
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import io
from PIL import Image


predidctions_df = pd.read_csv('/compass/Shared/Users/bogdanov/ml_tokamak/runs/24-05-27, 20-00-44 both, finer lr_scheduler, 3 classes, resnet34_all_layers/prediction_df.csv', index_col=0)

In [2]:
predidctions_df

,shot,prediction,label,time,prob_0,prob_1,prob_2
0,16534,0,0,960.2,1.000000,3.376573e-08,8.351586e-08
1,16534,0,0,960.4,0.999934,5.506525e-06,6.088502e-05
2,16534,0,0,960.6,0.999980,1.946953e-05,1.545336e-07
3,16534,0,0,960.8,0.999846,1.525366e-04,1.709923e-06
4,16534,0,0,961.0,0.999998,1.859756e-06,4.890233e-07
...,...,...,...,...,...,...,...
109205,19379,0,0,1329.0,1.000000,7.890306e-09,1.695548e-10
109206,19379,0,0,1329.2,0.999991,9.325103e-06,2.766426e-10
109207,19379,0,0,1329.4,0.999997,2.654907e-06,2.526000e-07
109208,19379,0,0,1329.6,1.000000,1.639063e-07,1.729383e-10


In [9]:
thr = 0.6

def group_continuous_elms(df, thr=0.5):
    """
    Group consecutive ELM periods by shot and time continuity.
    Returns a list of tuples (shot, start_idx, end_idx, is_detected)
    """
    elm_groups = []
    
    for shot in df['shot'].unique():
        shot_df = df[df['shot'] == shot].sort_values('time').reset_index(drop=True)
        
        # Find continuous ELM periods (label = 2)
        elm_mask = shot_df['label'] == 2
        if not elm_mask.any():
            continue
            
        # Find start and end of each continuous ELM period
        elm_changes = elm_mask.diff().fillna(False)
        elm_starts = shot_df.index[elm_changes & elm_mask].tolist()
        elm_ends = shot_df.index[elm_changes & ~elm_mask].tolist()
        
        # Handle case where ELM period extends to end of shot
        if elm_mask.iloc[-1] and (not elm_ends or elm_ends[-1] < elm_starts[-1]):
            elm_ends.append(len(shot_df))
        
        # Process each ELM group
        for start, end in zip(elm_starts, elm_ends):
            elm_group = shot_df.iloc[start:end]
            total_elms = len(elm_group)
            correct_predictions = (elm_group['prediction'] == 2).sum()
            
            # Check if at least 2/3 are correctly predicted
            is_detected = correct_predictions >= (thr * total_elms)
            elm_groups.append((shot, start, end, is_detected, total_elms, correct_predictions))
    
    return elm_groups

# Group continuous ELMs
elm_groups = group_continuous_elms(predidctions_df, thr=thr)

# Calculate metrics
total_elm_groups = len(elm_groups)
detected_elm_groups = sum(1 for group in elm_groups if group[3])  # group[3] is is_detected

print(f"Total continuous ELM groups found: {total_elm_groups}")
print(f"ELM groups detected (≥{thr:.1f} correct): {detected_elm_groups}")

# Calculate precision, recall, and F1
# For this analysis, we consider:
# - True Positives: ELM groups that were detected (≥2/3 correct predictions)
# - False Negatives: ELM groups that were not detected (<2/3 correct predictions)
# - False Positives: Need to find predicted ELM groups that don't correspond to actual ELMs

def group_predicted_elms(df, thr=0.5):
    """Group consecutive predicted ELM periods."""
    predicted_elm_groups = []
    
    for shot in df['shot'].unique():
        shot_df = df[df['shot'] == shot].sort_values('time').reset_index(drop=True)
        
        # Find continuous predicted ELM periods (prediction = 2)
        pred_elm_mask = shot_df['prediction'] == 2
        if not pred_elm_mask.any():
            continue
            
        # Find start and end of each continuous predicted ELM period
        pred_changes = pred_elm_mask.diff().fillna(False)
        pred_starts = shot_df.index[pred_changes & pred_elm_mask].tolist()
        pred_ends = shot_df.index[pred_changes & ~pred_elm_mask].tolist()
        
        # Handle case where predicted ELM extends to end
        if pred_elm_mask.iloc[-1] and (not pred_ends or pred_ends[-1] < pred_starts[-1]):
            pred_ends.append(len(shot_df))
        
        # Check each predicted group against actual labels
        for start, end in zip(pred_starts, pred_ends):
            if end - start < 2:
                continue  # Skip very short predictions
            pred_group = shot_df.iloc[start:end]
            actual_elms = (pred_group['label'] == 2).sum()
            total_predictions = len(pred_group)
            
            # Consider it a true positive if ≥2/3 of the predicted group overlaps with actual ELMs
            is_true_positive = actual_elms >= (thr * total_predictions)
            predicted_elm_groups.append((shot, start, end, is_true_positive, total_predictions, actual_elms))
    
    return predicted_elm_groups

predicted_elm_groups = group_predicted_elms(predidctions_df, thr=thr)

# Calculate final metrics
true_positives = sum(1 for group in predicted_elm_groups if group[3])
false_positives = len(predicted_elm_groups) - true_positives
false_negatives = total_elm_groups - detected_elm_groups

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nContinuous ELM Detection Metrics:")
print(f"True Positives: {true_positives}")
print(f"False Positives: {false_positives}")
print(f"False Negatives: {false_negatives}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1_score:.3f}")

# Additional analysis: distribution of ELM group sizes and detection rates
elm_group_info = pd.DataFrame(elm_groups, columns=['shot', 'start', 'end', 'detected', 'total_elms', 'correct_preds'])
elm_group_info['group_size'] = elm_group_info['end'] - elm_group_info['start']
elm_group_info['detection_rate'] = elm_group_info['correct_preds'] / elm_group_info['total_elms']

print(f"\nELM Group Statistics:")
print(f"Average ELM group size: {elm_group_info['group_size'].mean():.1f}")
print(f"Median ELM group size: {elm_group_info['group_size'].median():.1f}")
print(f"Average detection rate within groups: {elm_group_info['detection_rate'].mean():.3f}")

# Show detection rate by group size
size_analysis = elm_group_info.groupby('group_size').agg({
    'detected': ['count', 'sum', 'mean']
}).round(3)
print(f"\nDetection rate by ELM group size:")
print(size_analysis.head(10))

/tmp/ipykernel_1949635/3174296944.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  elm_changes = elm_mask.diff().fillna(False)
/tmp/ipykernel_1949635/3174296944.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pred_changes = pred_elm_mask.diff().fillna(False)


Total continuous ELM groups found: 847
ELM groups detected (≥0.6 correct): 720

Continuous ELM Detection Metrics:
True Positives: 909
False Positives: 122
False Negatives: 127
Precision: 0.882
Recall: 0.877
F1 Score: 0.880

ELM Group Statistics:
Average ELM group size: 8.4
Median ELM group size: 7.0
Average detection rate within groups: 0.809

Detection rate by ELM group size:
           detected            
              count  sum   mean
group_size                     
1                10    0  0.000
2                15    2  0.133
3                16   14  0.875
4                74   61  0.824
5               134  133  0.993
6               148  140  0.946
7               166  151  0.910
8                57   47  0.825
9                 6    6  1.000
10               37   29  0.784


In [4]:
def calculate_elm_detection_metrics(df, thr=0.6):
    """
    Calculate ELM detection metrics with proper overlap handling.
    """
    # Get actual and predicted ELM groups
    actual_groups = group_continuous_elms(df, thr=0.0)  # All actual ELM groups
    predicted_groups = group_predicted_elms(df, thr=0.0)  # All predicted ELM groups
    
    # Convert to more manageable format with shot-time ranges
    actual_ranges = []
    for shot, start, end, _, _, _ in actual_groups:
        shot_df = df[df['shot'] == shot].sort_values('time').reset_index(drop=True)
        time_start = shot_df.iloc[start]['time']
        time_end = shot_df.iloc[end-1]['time'] if end < len(shot_df) else shot_df.iloc[-1]['time']
        actual_ranges.append((shot, time_start, time_end, start, end))
    
    predicted_ranges = []
    for shot, start, end, _, _, _ in predicted_groups:
        shot_df = df[df['shot'] == shot].sort_values('time').reset_index(drop=True)
        time_start = shot_df.iloc[start]['time']
        time_end = shot_df.iloc[end-1]['time'] if end < len(shot_df) else shot_df.iloc[-1]['time']
        predicted_ranges.append((shot, time_start, time_end, start, end))
    
    # Calculate overlaps and determine TP, FP, FN
    true_positives = 0
    false_positives = 0
    matched_actual = set()
    
    for pred_shot, pred_start_time, pred_end_time, pred_start_idx, pred_end_idx in predicted_ranges:
        # Check if this predicted group overlaps significantly with any actual group
        is_true_positive = False
        
        for i, (act_shot, act_start_time, act_end_time, act_start_idx, act_end_idx) in enumerate(actual_ranges):
            if pred_shot != act_shot:
                continue
                
            # Calculate overlap
            overlap_start = max(pred_start_time, act_start_time)
            overlap_end = min(pred_end_time, act_end_time)
            
            if overlap_start < overlap_end:  # There is overlap
                overlap_duration = overlap_end - overlap_start
                pred_duration = pred_end_time - pred_start_time
                act_duration = act_end_time - act_start_time
                
                # Consider TP if overlap is significant (≥threshold of both periods)
                if (overlap_duration >= thr * pred_duration and 
                    overlap_duration >= thr * act_duration):
                    is_true_positive = True
                    matched_actual.add(i)
                    break
        
        if is_true_positive:
            true_positives += 1
        else:
            false_positives += 1
    
    # False negatives are actual groups that weren't matched
    false_negatives = len(actual_ranges) - len(matched_actual)
    
    # Calculate metrics
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'total_actual_groups': len(actual_ranges),
        'total_predicted_groups': len(predicted_ranges)
    }

# Use the improved calculation
metrics = calculate_elm_detection_metrics(predidctions_df, thr=thr)

print(f"Improved Continuous ELM Detection Metrics:")
print(f"Total actual ELM groups: {metrics['total_actual_groups']}")
print(f"Total predicted ELM groups: {metrics['total_predicted_groups']}")
print(f"True Positives: {metrics['true_positives']}")
print(f"False Positives: {metrics['false_positives']}")
print(f"False Negatives: {metrics['false_negatives']}")
print(f"Precision: {metrics['precision']:.3f}")
print(f"Recall: {metrics['recall']:.3f}")
print(f"F1 Score: {metrics['f1_score']:.3f}")

/tmp/ipykernel_1949635/4237821253.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  elm_changes = elm_mask.diff().fillna(False)
/tmp/ipykernel_1949635/4237821253.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pred_changes = pred_elm_mask.diff().fillna(False)


Improved Continuous ELM Detection Metrics:
Total actual ELM groups: 847
Total predicted ELM groups: 1526
True Positives: 481
False Positives: 1045
False Negatives: 366
Precision: 0.315
Recall: 0.568
F1 Score: 0.405
